# Capy – server model mở trên Colab

Chạy model thị giác **mã nguồn mở** (Qwen3-VL qua Ollama) thay cho OpenAI, để ảnh slide không đi tới model đóng.

1. **Runtime → Change runtime type → T4 GPU**, rồi chạy lần lượt từng ô.
2. Ô cuối in ra 2 dòng `OPENAI_BASE_URL` và `OPENAI_MODEL` → dán vào `.env` của repo, chạy lại `npm run dev`.

Lưu ý:
- Ảnh vẫn đi qua máy ảo Google Colab, nhưng không qua OpenAI/Anthropic và không dùng để train. Dùng thật trong công ty: chạy **đúng các lệnh này trên máy của công ty**, chỉ đổi `OPENAI_BASE_URL`.
- URL `trycloudflare.com` là công khai, ai biết URL đều gọi được model → không chia sẻ URL, tắt Colab khi xong.
- Colab miễn phí tự ngắt khi để lâu không dùng; ngắt thì chạy lại từ đầu (URL sẽ đổi).


In [ ]:
# Model: qwen3-vl:8b (~6 GB, đọc tiếng Việt tốt hơn). Chậm/thiếu RAM → đổi sang "qwen2.5vl:7b" hoặc "qwen3-vl:4b".
MODEL = "qwen3-vl:8b"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Cài Ollama
!sudo apt-get -qq update && sudo apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Chạy Ollama nền, giữ model trong GPU
import os, subprocess, time
env = {**os.environ, "OLLAMA_HOST": "0.0.0.0:11434", "OLLAMA_KEEP_ALIVE": "-1"}
subprocess.Popen(["ollama", "serve"], env=env, stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
!ollama --version

In [ ]:
# Tải model (vài phút)
!ollama pull {MODEL}

In [ ]:
# Thử nhanh qua API chuẩn OpenAI (giống cách app gọi) + nạp sẵn model vào GPU
import requests
r = requests.post("http://localhost:11434/v1/chat/completions", json={
    "model": MODEL,
    "messages": [{"role": "user", "content": "Chào bằng một câu tiếng Việt."}],
    "max_tokens": 50,
}, timeout=600)
print(r.json()["choices"][0]["message"]["content"])

In [ ]:
# Mở đường hầm công khai tới Ollama (cloudflared, không cần tài khoản)
import re
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:11434"], stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(30):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        url = m.group(0)
        break
print("Dán vào .env của repo:\n")
print(f"OPENAI_BASE_URL={url}/v1")
print(f"OPENAI_MODEL={MODEL}")

Giữ tab Colab mở trong lúc demo. Xem log: `!tail ollama.log tunnel.log`.